# Pipeline Debug Notebook
End-to-end debug of: `train_model` → `evaluate_model` (ingestion) → `score` (scoring)

Run cells top-to-bottom. Each section shows intermediate outputs so you can spot errors at each stage.

In [ ]:
import sys
import json
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
from xml.etree.ElementTree import parse as parse_xml

# Paths
ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT / 'ingestion_program'))
sys.path.insert(0, str(ROOT / 'solution'))
sys.path.insert(0, str(ROOT / 'scoring_program'))

DATA_DIR        = ROOT / 'dev_phase' / 'input_data'
REFERENCE_DIR   = ROOT / 'dev_phase' / 'reference_data'
INGESTION_OUT   = ROOT / 'ingestion_res'
SCORING_OUT     = ROOT / 'scoring_res'

INGESTION_OUT.mkdir(parents=True, exist_ok=True)
SCORING_OUT.mkdir(parents=True, exist_ok=True)

print('ROOT        :', ROOT)
print('DATA_DIR    :', DATA_DIR)
print('REFERENCE   :', REFERENCE_DIR)
print('PyTorch     :', torch.__version__)
print('CUDA        :', torch.cuda.is_available())

---
## 1. Train Model

In [ ]:
from submission import train_model

training_dir = DATA_DIR / 'train'
print(f'Training directory: {training_dir}')
print(f'Contents: {list(training_dir.iterdir())}')

print('\n--- Starting training ---')
t0 = time.time()
model = train_model(str(training_dir))
train_time = time.time() - t0

print(f'\n✓ Training complete in {train_time:.1f}s')
print(f'Model type: {type(model)}')

---
## 2. Run Inference on a Few Frames
Quick sanity check — run the model on individual images and visualise the predictions.

In [ ]:
from tokam2d_utils import TokamDataset

# Load a small slice of the training set so we can inspect predictions
train_dataset = TokamDataset(training_dir, include_unlabeled=False)
print(f'Dataset size: {len(train_dataset)}')

model.eval()

NUM_SAMPLES = 6
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for plot_idx in range(NUM_SAMPLES):
    img_tensor, target = train_dataset[plot_idx]

    with torch.no_grad():
        preds = model((img_tensor,))  # model expects a tuple
    pred = preds[0]

    # Display image
    img_np = img_tensor.squeeze().numpy()
    img_norm = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
    axes[plot_idx].imshow(img_norm, cmap='gray')
    axes[plot_idx].axis('off')

    # Ground truth (green)
    if target['boxes'] is not None and len(target['boxes']) > 0:
        for box in target['boxes']:
            b = box.numpy() if hasattr(box, 'numpy') else np.array(box)
            h, w = img_np.shape
            if b[0] > 1:  # pixel coords
                x1, y1, bw, bh = b[0], b[1], b[2], b[3]
            else:          # normalised
                x1, y1, bw, bh = b[0]*w, b[1]*h, b[2]*w, b[3]*h
            rect = patches.Rectangle((x1, y1), bw, bh,
                                      linewidth=2, edgecolor='lime', facecolor='none')
            axes[plot_idx].add_patch(rect)

    # Predictions (red)
    det_count = 0
    if pred['boxes'] is not None and len(pred['boxes']) > 0:
        for box, score in zip(pred['boxes'], pred['scores']):
            x1, y1, x2, y2 = box.numpy()
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                      linewidth=2, edgecolor='red',
                                      linestyle='--', facecolor='none')
            axes[plot_idx].add_patch(rect)
            axes[plot_idx].text(x1, y1-4, f'{score:.2f}', color='red',
                                fontsize=9, weight='bold',
                                bbox=dict(facecolor='black', alpha=0.5, pad=1))
            det_count += 1

    gt_count = len(target['boxes']) if target['boxes'] is not None else 0
    fi = target.get('frame_index', plot_idx)
    axes[plot_idx].set_title(f'{fi} | GT:{gt_count}  Pred:{det_count}', fontsize=10)

from matplotlib.patches import Patch
fig.legend(handles=[
    Patch(facecolor='none', edgecolor='lime', linewidth=2, label='Ground Truth'),
    Patch(facecolor='none', edgecolor='red',  linewidth=2, linestyle='--', label='Prediction')
], loc='upper center', ncol=2, fontsize=12)
plt.suptitle('Direct Model Inference (Training Frames)', fontsize=14, weight='bold', y=0.98)
plt.tight_layout()
plt.show()

---
## 3. Run Ingestion (evaluate_model)
Reproduces `ingestion.py` locally and writes prediction XMLs.

In [ ]:
from tokam2d_utils.xml_loader import dump_to_xml

EVAL_SETS = ['test']
# Add 'private_test' if that directory exists
if (DATA_DIR / 'private_test').exists():
    EVAL_SETS.append('private_test')

def collate_fn(batch):
    return tuple(zip(*batch))

def evaluate_model(model, data_dir):
    eval_dataset = TokamDataset(data_dir)
    eval_dataloader = torch.utils.data.DataLoader(
        eval_dataset, batch_size=2, collate_fn=collate_fn
    )
    model.eval()
    res = []
    for X, y in eval_dataloader:
        with torch.no_grad():
            y_pred = model(X)
        y_pred = [
            {**y_p, 'frame_index': y_t['frame_index']}
            for y_p, y_t in zip(y_pred, y)
        ]
        res.extend(y_pred)
    return res

all_results = {}
t0 = time.time()
for eval_set in EVAL_SETS:
    eval_dir = DATA_DIR / eval_set
    if not eval_dir.exists():
        print(f'⚠ Skipping {eval_set} — directory not found: {eval_dir}')
        continue
    print(f'Evaluating {eval_set} ...')
    results = evaluate_model(model, eval_dir)
    all_results[eval_set] = results
    out_xml = INGESTION_OUT / f'{eval_set}_predictions.xml'
    dump_to_xml(results, out_xml)
    print(f'  → {len(results)} frames written to {out_xml}')
test_time = time.time() - t0

# Write metadata
meta = dict(train_time=train_time, test_time=test_time)
with open(INGESTION_OUT / 'metadata.json', 'w') as f:
    json.dump(meta, f)

print(f'\n✓ Ingestion done in {test_time:.1f}s')
print(f'metadata: {meta}')

---
## 4. Inspect Prediction XML
Parse and show what was written to the XML file.

In [ ]:
for eval_set in EVAL_SETS:
    xml_path = INGESTION_OUT / f'{eval_set}_predictions.xml'
    if not xml_path.exists():
        print(f'⚠ {xml_path} not found')
        continue

    tree = parse_xml(xml_path)
    root = tree.getroot()
    images = root.findall('image')

    total_boxes = sum(len(img.findall('box')) for img in images)
    frames_with_det = sum(1 for img in images if len(img.findall('box')) > 0)

    print(f'\n=== {eval_set}_predictions.xml ===')
    print(f'  Total frames  : {len(images)}')
    print(f'  Frames w/ detections: {frames_with_det}')
    print(f'  Total boxes   : {total_boxes}')

    if total_boxes == 0:
        print('  ⚠ WARNING: No detections at all — model may not have converged')

    print('\n  First 10 frames:')
    for img in images[:10]:
        boxes = img.findall('box')
        scores = [float(b.attrib.get('score', 0)) for b in boxes]
        score_str = f'scores={[f"{s:.3f}" for s in scores]}' if scores else 'no detections'
        print(f'    {img.attrib["index"]:25s}  {len(boxes)} boxes  {score_str}')

---
## 5. Compare Predictions vs Ground Truth (Visual)
Side-by-side visualisation on test frames.

In [ ]:
eval_set = 'test'
pred_xml_path = INGESTION_OUT / f'{eval_set}_predictions.xml'

# Load predictions from XML
def read_xml_preds(path):
    tree = parse_xml(path)
    root = tree.getroot()
    out = {}
    for img in root.findall('image'):
        idx = img.attrib.get('index', img.attrib.get('name', '?'))
        boxes = []
        scores = []
        for b in img.findall('box'):
            boxes.append((
                float(b.attrib['xtl']), float(b.attrib['ytl']),
                float(b.attrib['xbr']), float(b.attrib['ybr'])
            ))
            scores.append(float(b.attrib.get('score', 1.0)))
        out[idx] = {'boxes': boxes, 'scores': scores}
    return out

if not pred_xml_path.exists():
    print(f'⚠ {pred_xml_path} not found — run ingestion first (cell 4)')
else:
    predictions = read_xml_preds(pred_xml_path)

    # Load test frames
    test_dir = DATA_DIR / eval_set
    test_dataset = TokamDataset(test_dir, include_unlabeled=True)
    NUM_SHOW = min(6, len(test_dataset))

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()

    for plot_idx in range(NUM_SHOW):
        img_tensor, target = test_dataset[plot_idx]
        frame_idx = str(target['frame_index'])

        img_np = img_tensor.squeeze().numpy()
        img_norm = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
        axes[plot_idx].imshow(img_norm, cmap='gray')
        axes[plot_idx].axis('off')

        # Predictions from XML
        frame_key = frame_idx.split('-')[-1] if '-' in frame_idx else frame_idx
        full_key = next((k for k in predictions if k == frame_idx), None) or \
                   next((k for k in predictions if frame_key in k), None)

        pred_count = 0
        if full_key and predictions[full_key]['boxes']:
            for (x1, y1, x2, y2), score in zip(
                predictions[full_key]['boxes'],
                predictions[full_key]['scores']
            ):
                rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                          linewidth=2, edgecolor='red',
                                          linestyle='--', facecolor='none')
                axes[plot_idx].add_patch(rect)
                axes[plot_idx].text(x1, y1-4, f'{score:.2f}',
                                    color='red', fontsize=9, weight='bold',
                                    bbox=dict(facecolor='black', alpha=0.5, pad=1))
                pred_count += 1

        axes[plot_idx].set_title(f'{frame_idx} | Pred:{pred_count}', fontsize=10)

    plt.suptitle('Predictions on Test Frames (from XML)', fontsize=14, weight='bold')
    plt.tight_layout()
    plt.show()
    print(f'Total prediction keys in XML: {len(predictions)}')

---
## 6. Run Scoring
Computes AP50 using IoMean. Requires reference data in `dev_phase/reference_data/`.

In [ ]:
from scoring import compute_ap, read_xml

if not REFERENCE_DIR.exists():
    print(f'⚠ Reference directory not found: {REFERENCE_DIR}')
    print('  Create dev_phase/reference_data/ with ground truth XML files to enable scoring.')
else:
    scores = {}
    for eval_set in EVAL_SETS:
        pred_path = INGESTION_OUT / f'{eval_set}_predictions.xml'
        ref_path  = REFERENCE_DIR / f'{eval_set}_labels.xml'

        if not pred_path.exists():
            print(f'⚠ Missing predictions: {pred_path}')
            continue
        if not ref_path.exists():
            print(f'⚠ Missing reference:   {ref_path}')
            continue

        predictions = read_xml(pred_path)
        targets     = read_xml(ref_path)

        print(f'\n=== {eval_set} ===')
        print(f'  Prediction frames : {len(predictions)}')
        print(f'  Reference frames  : {len(targets)}')

        matched = set(predictions.keys()) & set(targets.keys())
        print(f'  Matched frames    : {len(matched)}')

        ap50 = compute_ap(predictions, targets, threshold=0.5)
        scores[eval_set] = ap50
        print(f'  AP50              : {ap50:.4f}')

    if scores:
        meta = json.loads((INGESTION_OUT / 'metadata.json').read_text())
        scores.update(**meta)
        print('\nFinal scores:', json.dumps(scores, indent=2))

        SCORING_OUT.mkdir(parents=True, exist_ok=True)
        (SCORING_OUT / 'scores.json').write_text(json.dumps(scores))
        print(f'\n✓ Written to {SCORING_OUT / "scores.json"}')

---
## 7. Score Breakdown per Frame
Shows per-frame IoMean between predictions and ground truth.

In [ ]:
from discopat.metrics import compute_iomean

eval_set = 'test'
pred_path = INGESTION_OUT / f'{eval_set}_predictions.xml'
ref_path  = REFERENCE_DIR / f'{eval_set}_labels.xml' if REFERENCE_DIR.exists() else None

if not pred_path.exists():
    print(f'⚠ Run ingestion (cell 4) first.')
elif ref_path is None or not ref_path.exists():
    print(f'⚠ No reference labels found at {ref_path}')
else:
    predictions = read_xml(pred_path)
    targets     = read_xml(ref_path)

    frame_results = []
    for frame_key, target in targets.items():
        gt_boxes = target['boxes']
        if frame_key not in predictions or not predictions[frame_key]['boxes']:
            frame_results.append({'frame': frame_key, 'gt': len(gt_boxes),
                                   'pred': 0, 'best_iomean': 0.0})
            continue
        pred_boxes = predictions[frame_key]['boxes']
        # Best IoMean among all pred-gt pairs
        best = max(
            (compute_iomean(p, g) for p in pred_boxes for g in gt_boxes),
            default=0.0
        )
        frame_results.append({'frame': frame_key, 'gt': len(gt_boxes),
                               'pred': len(pred_boxes), 'best_iomean': best})

    frame_results.sort(key=lambda x: x['best_iomean'])

    print(f'Per-frame results ({eval_set}):'
          f'\n{"Frame":30s} {"GT":>4} {"Pred":>5} {"Best IoMean":>12}')
    print('-' * 55)
    for r in frame_results:
        tp_flag = '✓' if r['best_iomean'] >= 0.5 else '✗'
        print(f"{r['frame']:30s} {r['gt']:>4} {r['pred']:>5} "
              f"{r['best_iomean']:>12.3f}  {tp_flag}")

    tp_count = sum(1 for r in frame_results if r['best_iomean'] >= 0.5)
    print(f'\nFrames with IoMean >= 0.5: {tp_count}/{len(frame_results)}')

---
## 8. Confidence Score Distribution
Histogram of all prediction confidence scores.

In [ ]:
for eval_set in EVAL_SETS:
    pred_path = INGESTION_OUT / f'{eval_set}_predictions.xml'
    if not pred_path.exists():
        continue

    tree = parse_xml(pred_path)
    root = tree.getroot()
    all_scores = [
        float(b.attrib.get('score', 1.0))
        for img in root.findall('image')
        for b in img.findall('box')
    ]

    if not all_scores:
        print(f'{eval_set}: No detections to show')
        continue

    print(f'{eval_set}: {len(all_scores)} total detections')
    print(f'  min={min(all_scores):.3f}  max={max(all_scores):.3f}  '
          f'mean={np.mean(all_scores):.3f}  median={np.median(all_scores):.3f}')

    plt.figure(figsize=(10, 4))
    plt.hist(all_scores, bins=50, color='steelblue', edgecolor='white')
    plt.axvline(0.25, color='orange', linestyle='--', label='default YOLO conf=0.25')
    plt.axvline(0.5,  color='red',    linestyle='--', label='conf=0.5')
    plt.xlabel('Confidence Score')
    plt.ylabel('Count')
    plt.title(f'Confidence Distribution — {eval_set}')
    plt.legend()
    plt.tight_layout()
    plt.show()

---
## 9. Checklist Summary

In [ ]:
print('=' * 60)
print('PIPELINE CHECKLIST')
print('=' * 60)

checks = [
    ('Model trained',         model is not None),
    ('test evaluation run',   (INGESTION_OUT / 'test_predictions.xml').exists()),
    ('metadata.json written', (INGESTION_OUT / 'metadata.json').exists()),
    ('Reference data exists', REFERENCE_DIR.exists()),
    ('scores.json written',   (SCORING_OUT / 'scores.json').exists()),
]

for label, ok in checks:
    status = '✓' if ok else '✗'
    print(f'  {status}  {label}')

if (SCORING_OUT / 'scores.json').exists():
    final = json.loads((SCORING_OUT / 'scores.json').read_text())
    print(f'\nFinal scores:')
    for k, v in final.items():
        print(f'  {k:20s}: {v}')

# Detections summary
print()
for eval_set in EVAL_SETS:
    p = INGESTION_OUT / f'{eval_set}_predictions.xml'
    if p.exists():
        tree = parse_xml(p)
        total_boxes = sum(len(img.findall('box')) for img in tree.getroot().findall('image'))
        frames = len(tree.getroot().findall('image'))
        print(f'  {eval_set}: {total_boxes} total detections across {frames} frames')
        if total_boxes == 0:
            print(f'  ⚠ No detections — check training convergence or conf threshold')